# Yahoo Finance Stock Data Tool  (RapidAPI — US + India)

Fetches live financial data via **Yahoo Finance on RapidAPI**.

## Supported input formats

| What you type | How it works |
|---|---|
| `AAPL` | Tries US market directly ✓ |
| `RELIANCE.NS` | Tries India NSE directly ✓ |
| `TCS.BO` | Tries India BSE directly ✓ |
| `SUZLON` | Tries US → no data → auto-retries as `SUZLON.NS` → `SUZLON.BO` ✓ |
| `WIPRO` | Tries US → no data → auto-retries as `WIPRO.NS` ✓ |

## Auto-fallback chain (for bare tickers with no suffix)
```
SUZLON
  └─ try US  (region=US)       → no price found
       └─ try NSE (symbol.NS)  → price found ✓  return India data
            └─ try BSE (symbol.BO) → if NSE also empty
```

## Response note
Yahoo Finance wraps numbers as `{"raw": 52.4, "fmt": "52.40"}`.  
The `_raw()` helper unwraps them transparently.

In [ ]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv()

_RAPIDAPI_KEY  = os.getenv("RAPIDAPI_KEY")
_RAPIDAPI_HOST = "yahoo-finance166.p.rapidapi.com"
_HEADERS = {
    "x-rapidapi-key":  _RAPIDAPI_KEY,
    "x-rapidapi-host": _RAPIDAPI_HOST,
    "Content-Type":    "application/json",
}


# ── Helpers ────────────────────────────────────────────────────────────────

def _raw(obj: dict, *keys):
    """
    Navigate a nested dict and unwrap Yahoo Finance's {"raw": v} envelope.
    Returns None safely if any key is missing at any level.
    """
    for key in keys:
        if not isinstance(obj, dict):
            return None
        obj = obj.get(key)
    return obj.get("raw") if isinstance(obj, dict) else obj


def _fetch_summary(symbol: str, region: str) -> dict:
    """
    Single API call to Yahoo Finance get-summary.
    Returns the parsed inner data dict (may be empty {} if ticker not found).
    """
    url    = f"https://{_RAPIDAPI_HOST}/api/stock/get-summary"
    params = {"symbol": symbol, "region": region, "lang": "en-US"}

    try:
        with httpx.Client(timeout=20.0) as client:
            resp = client.get(url, headers=_HEADERS, params=params)
            resp.raise_for_status()
            raw = resp.json()
    except Exception:
        return {}  # treat any fetch error as "not found" so fallback can proceed

    # Unwrap {"result": [...], "error": null} envelope if present
    if isinstance(raw, dict) and "result" in raw:
        results = raw["result"]
        return results[0] if results else {}
    return raw if isinstance(raw, dict) else {}


def _parse_data(data: dict, symbol: str, region: str) -> dict:
    """
    Extract the flat metrics dict from a Yahoo Finance quoteSummary response.
    """
    price_mod = data.get("price", {})
    summary   = data.get("summaryDetail", {})
    fin       = data.get("financialData", {})
    stats     = data.get("defaultKeyStatistics", {})
    profile   = data.get("assetProfile", {})
    currency  = "₹" if region == "IN" else "$"

    return {
        "ticker":               symbol,
        "market":               "India" if region == "IN" else "US",
        "currency":             currency,
        "company_name":         price_mod.get("longName") or price_mod.get("shortName"),
        "current_price":        _raw(price_mod, "regularMarketPrice"),
        "previous_close":       _raw(price_mod, "regularMarketPreviousClose")
                                    or _raw(summary, "previousClose"),
        "pe_ratio":             _raw(summary,   "trailingPE"),
        "forward_pe":           _raw(summary,   "forwardPE") or _raw(stats, "forwardPE"),
        "price_to_book":        _raw(stats,     "priceToBook"),
        "revenue_growth":       _raw(fin,       "revenueGrowth"),
        "earnings_growth":      _raw(fin,       "earningsGrowth"),
        "market_cap":           _raw(price_mod, "marketCap") or _raw(summary, "marketCap"),
        "52_week_high":         _raw(summary,   "fiftyTwoWeekHigh"),
        "52_week_low":          _raw(summary,   "fiftyTwoWeekLow"),
        "dividend_yield":       _raw(summary,   "dividendYield"),
        "beta":                 _raw(summary,   "beta"),
        "sector":               profile.get("sector"),
        "industry":             profile.get("industry"),
        "analyst_target_price": _raw(fin,       "targetMeanPrice"),
        "recommendation":       _raw(fin,       "recommendationKey"),
    }


# ── Public API ─────────────────────────────────────────────────────────────

def get_stock_data(ticker: str) -> dict:
    """
    Fetch financial metrics for any stock ticker.

    Auto-fallback chain for tickers with NO exchange suffix:
      1. Try as US ticker                   (e.g. SUZLON  → US)
      2. If no price found → try as NSE     (e.g. SUZLON.NS → IN)
      3. If still no price → try as BSE     (e.g. SUZLON.BO → IN)

    Tickers WITH a suffix are routed directly (no fallback needed):
      RELIANCE.NS → IN immediately
      TCS.BO      → IN immediately

    Args:
        ticker: 'AAPL' / 'SUZLON' / 'RELIANCE.NS' / 'TCS.BO'

    Returns:
        Flat dict of financial metrics. Missing fields are None.
    """
    t = ticker.upper().strip()

    # ── Explicit suffix → direct lookup, no fallback needed ────────────────
    if t.endswith(".NS"):
        data = _fetch_summary(t, "IN")
        return _parse_data(data, t, "IN")

    if t.endswith(".BO"):
        data = _fetch_summary(t, "IN")
        return _parse_data(data, t, "IN")

    # ── No suffix → try US first ────────────────────────────────────────────
    print(f"[yfinance] Trying US market for '{t}'...")
    us_data = _fetch_summary(t, "US")
    us_result = _parse_data(us_data, t, "US")

    if us_result.get("current_price") is not None:
        print(f"[yfinance] Found '{t}' on US market")
        return us_result

    # ── US returned no price → try India NSE (.NS) ─────────────────────────
    ns_symbol = f"{t}.NS"
    print(f"[yfinance] '{t}' not found on US, trying {ns_symbol} (India NSE)...")
    ns_data   = _fetch_summary(ns_symbol, "IN")
    ns_result = _parse_data(ns_data, ns_symbol, "IN")

    if ns_result.get("current_price") is not None:
        print(f"[yfinance] Found '{ns_symbol}' on India NSE")
        return ns_result

    # ── NSE also empty → try India BSE (.BO) ──────────────────────────────
    bo_symbol = f"{t}.BO"
    print(f"[yfinance] '{ns_symbol}' not found, trying {bo_symbol} (India BSE)...")
    bo_data   = _fetch_summary(bo_symbol, "IN")
    bo_result = _parse_data(bo_data, bo_symbol, "IN")

    if bo_result.get("current_price") is not None:
        print(f"[yfinance] Found '{bo_symbol}' on India BSE")
        return bo_result

    # ── All three failed → return US result (will have None fields) ────────
    print(f"[yfinance] WARNING: no data found for '{t}' on US, NSE, or BSE")
    return us_result


In [ ]:
if __name__ == "__main__":
    import json

    test_tickers = [
        "AAPL",          # US — direct hit
        "SUZLON",        # India NSE — no suffix, needs auto-fallback
        "RELIANCE.NS",   # India NSE — explicit suffix
        "TCS.BO",        # India BSE — explicit suffix
        "WIPRO",         # India NSE — no suffix, needs auto-fallback
    ]

    for ticker in test_tickers:
        print(f"\n{'='*55}")
        print(f" Input: '{ticker}'")
        print(f"{'='*55}")
        result = get_stock_data(ticker)
        print(f" → Resolved as: {result['ticker']}  ({result['market']})")
        print(f" → Company    : {result['company_name']}")
        print(f" → Price      : {result['currency']}{result['current_price']}")
        print(f" → Market Cap : {result['currency']}{result['market_cap']}")
